# Exploratory Environmental and Vector Analysis Pipeline (Fixed)
## Automatic discovery and analysis of all data in the CDM
### Now correctly links point vector data to parent regions via parent_location_code

### 1. Setup and Imports

In [22]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text
import geopandas as gpd
from shapely import wkt
import contextily as ctx
import warnings
import os
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from database.session import engine
from database.models import Location, VectorObservation, EnvironmentalFactor

# Global output path for figures
FIGURE_OUTPUT_PATH = "/mnt/monadworld/projects/phd/article_manuscripts/new/figures/exploratory"

# Create the directory if it doesn't exist
os.makedirs(FIGURE_OUTPUT_PATH, exist_ok=True)
print(f"Figures will be saved to: {FIGURE_OUTPUT_PATH}")

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

Figures will be saved to: /mnt/monadworld/projects/phd/article_manuscripts/new/figures/exploratory


### 2. Discovery Functions – Find All Available Data (Fixed for Hierarchy)

In [23]:
def get_available_countries():
    """Return DataFrame of distinct countries with location data."""
    query = """
    SELECT DISTINCT country_name, country_code 
    FROM location 
    WHERE country_name IS NOT NULL AND country_name != ''
    ORDER BY country_name
    """
    return pd.read_sql(query, engine)

def get_available_regions(country_code):
    """Return DataFrame of regions that have environmental data and at least one vector point child."""
    query = """
    SELECT DISTINCT l.name, l.location_code
    FROM location l
    WHERE l.country_code = :country_code
      AND l.admin_level IN ('region', 'district', 'state', 'province')
      AND l.geometry_wkt IS NOT NULL
      AND EXISTS (SELECT 1 FROM environmental_factor e WHERE e.location_code = l.location_code)
      AND EXISTS (
          SELECT 1 FROM vector_observation v 
          JOIN location child ON v.location_code = child.location_code
          WHERE child.parent_location_code = l.location_code
      )
    ORDER BY l.name
    """
    return pd.read_sql(text(query), engine, params={'country_code': country_code})

def get_available_species():
    """Return list of distinct vector species present in the database."""
    query = "SELECT DISTINCT vector_species FROM vector_observation WHERE vector_species IS NOT NULL"
    df = pd.read_sql(query, engine)
    return df['vector_species'].tolist()

def get_global_date_range():
    """Return min and max date across environmental and vector data."""
    query_env = "SELECT MIN(date_id) as min_date, MAX(date_id) as max_date FROM environmental_factor"
    query_vec = "SELECT MIN(date_id) as min_date, MAX(date_id) as max_date FROM vector_observation"
    env_dates = pd.read_sql(query_env, engine)
    vec_dates = pd.read_sql(query_vec, engine)
    min_date = min(env_dates['min_date'].iloc[0], vec_dates['min_date'].iloc[0])
    max_date = max(env_dates['max_date'].iloc[0], vec_dates['max_date'].iloc[0])
    return min_date, max_date

def get_region_code(country_code, region_name):
    """Return location_code for a given region name and country."""
    query = "SELECT location_code FROM location WHERE country_code = :country_code AND name = :region_name"
    df = pd.read_sql(text(query), engine, params={'country_code': country_code, 'region_name': region_name})
    if df.empty:
        return None
    return df.iloc[0]['location_code']

### 3. Data Loading Functions (Fixed for Parent-Child Relationship)

In [24]:
def load_vector_data_for_region(region_code, species_pattern, start_date=None, end_date=None):
    """Load all vector observations whose location has parent_location_code = region_code."""
    query = """
    SELECT 
        v.date_id,
        v.location_code,
        v.vector_species,
        v.count,
        l.centroid_lat AS lat,
        l.centroid_lon AS lon,
        l.geometry_wkt AS geom_wkt,
        l.name AS location_name,
        l.admin_level,
        l.country_name,
        l.country_code
    FROM vector_observation v
    JOIN location l ON v.location_code = l.location_code
    WHERE l.parent_location_code = :region_code
      AND v.vector_species LIKE :species
    """
    params = {'region_code': region_code, 'species': species_pattern}
    if start_date:
        query += " AND v.date_id >= :start_date"
        params['start_date'] = start_date
    if end_date:
        query += " AND v.date_id <= :end_date"
        params['end_date'] = end_date
    query += " ORDER BY v.date_id"
    
    df_vec = pd.read_sql(text(query), engine, params=params)
    gdf_points = None
    if len(df_vec) > 0:
        df_with_coords = df_vec[df_vec['lat'].notna() & df_vec['lon'].notna()].copy()
        if len(df_with_coords) > 0:
            gdf_points = gpd.GeoDataFrame(
                df_with_coords,
                geometry=gpd.points_from_xy(df_with_coords['lon'], df_with_coords['lat']),
                crs='EPSG:4326'
            )
    return df_vec, gdf_points

def load_environmental_data(location_codes, start_date=None, end_date=None):
    """Load environmental data for specified location codes (region-level)."""
    if not location_codes:
        return pd.DataFrame()
    placeholders = ','.join([f':code_{i}' for i in range(len(location_codes))])
    params = {f'code_{i}': code for i, code in enumerate(location_codes)}
    query = f"""
    SELECT 
        e.date_id,
        e.location_code,
        e.temperature_mean,
        e.rainfall_total,
        l.name AS location_name,
        l.country_name,
        l.country_code
    FROM environmental_factor e
    JOIN location l ON e.location_code = l.location_code
    WHERE e.location_code IN ({placeholders})
    """
    if start_date:
        query += " AND e.date_id >= :start_date"
        params['start_date'] = start_date
    if end_date:
        query += " AND e.date_id <= :end_date"
        params['end_date'] = end_date
    query += " ORDER BY e.date_id"
    df_env = pd.read_sql(text(query), engine, params=params)
    if not df_env.empty:
        df_env['date'] = pd.to_datetime(df_env['date_id'])
    return df_env

### 4. Helper Functions: Aggregation, Merging

In [25]:
def aggregate_to_region_monthly(gdf_points, region_code):
    """Aggregate all vector points in the region to monthly totals."""
    if gdf_points is None or len(gdf_points) == 0:
        return pd.DataFrame()
    df_vec_monthly = gdf_points.groupby('date_id')['count'].agg(['sum', 'mean', 'std', 'count']).reset_index()
    df_vec_monthly.columns = ['date_id', 'total_count', 'mean_count', 'std_count', 'n_observations']
    df_vec_monthly['date'] = pd.to_datetime(df_vec_monthly['date_id'])
    df_vec_monthly['location_code'] = region_code
    return df_vec_monthly

def aggregate_environmental_monthly(df_env, region_code):
    """Aggregate daily environmental data to monthly means for a specific region."""
    if df_env is None or len(df_env) == 0:
        return pd.DataFrame()
    df_env_region = df_env[df_env['location_code'] == region_code].copy()
    if len(df_env_region) == 0:
        return pd.DataFrame()
    numeric_cols = ['temperature_mean', 'rainfall_total']
    df_env_indexed = df_env_region.set_index('date').copy()
    df_env_monthly = df_env_indexed[numeric_cols].resample('ME').agg(['mean', 'std', 'min', 'max'])
    df_env_monthly.columns = ['_'.join(col).strip() for col in df_env_monthly.columns.values]
    df_env_monthly = df_env_monthly.reset_index()
    df_env_monthly['date_id'] = df_env_monthly['date'].dt.strftime('%Y-%m-01')
    df_env_monthly['location_code'] = region_code
    return df_env_monthly

def merge_datasets(df_vec, df_env):
    """Merge vector and environmental datasets on date_id."""
    if df_vec.empty or df_env.empty:
        return pd.DataFrame()
    df_merged = pd.merge(df_vec, df_env, on='date_id', how='inner')
    if not df_merged.empty and 'date_x' in df_merged.columns:
        df_merged['date'] = df_merged['date_x']
    elif 'date' not in df_merged.columns:
        df_merged['date'] = pd.to_datetime(df_merged['date_id'])
    return df_merged

### 5. Plotting Functions (Save with Species and Region in Filename)

In [26]:
def plot_time_series(df_merged, region_name, species, output_path):
    if df_merged.empty:
        return None
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    axes[0].plot(df_merged['date'], df_merged['total_count'], marker='o', linestyle='-', linewidth=2, markersize=8, color='#2E86AB')
    if 'std_count' in df_merged.columns and not df_merged['std_count'].isna().all():
        axes[0].fill_between(df_merged['date'], df_merged['total_count']-df_merged['std_count'], df_merged['total_count']+df_merged['std_count'], alpha=0.2, color='#2E86AB')
    axes[0].set_ylabel('Abundance (count)', fontsize=12)
    axes[0].set_title(f'Monthly vector abundance - {region_name} - {species}', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    if 'temperature_mean_mean' in df_merged.columns and not df_merged['temperature_mean_mean'].isna().all():
        axes[1].plot(df_merged['date'], df_merged['temperature_mean_mean'], marker='s', linestyle='-', linewidth=2, markersize=8, color='#A23B72')
        axes[1].set_ylabel('Temperature (°C)', fontsize=12)
        axes[1].set_title('Monthly mean temperature', fontsize=14, fontweight='bold')
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, 'No temperature data available', ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
    if 'rainfall_total_mean' in df_merged.columns and not df_merged['rainfall_total_mean'].isna().all():
        axes[2].bar(df_merged['date'], df_merged['rainfall_total_mean'], width=20, color='#F18F01', alpha=0.7)
        axes[2].set_ylabel('Rainfall (mm)', fontsize=12)
        axes[2].set_title('Monthly total rainfall', fontsize=14, fontweight='bold')
        axes[2].grid(True, alpha=0.3)
    else:
        axes[2].text(0.5, 0.5, 'No rainfall data available', ha='center', va='center', transform=axes[2].transAxes, fontsize=12)
    plt.xlabel('Date', fontsize=12)
    plt.tight_layout()
    safe_species = species.replace('/', '_').replace(' ', '_')
    filepath = os.path.join(output_path, f'time_series_{region_name}_{safe_species}.png')
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return filepath

def analyze_correlations(df_merged, output_path, species):
    if df_merged.empty or len(df_merged) < 3:
        return None
    available_cols = ['total_count']
    if 'temperature_mean_mean' in df_merged.columns and not df_merged['temperature_mean_mean'].isna().all():
        available_cols.append('temperature_mean_mean')
    if 'rainfall_total_mean' in df_merged.columns and not df_merged['rainfall_total_mean'].isna().all():
        available_cols.append('rainfall_total_mean')
    if len(available_cols) < 2:
        return None
    corr_matrix = df_merged[available_cols].corr()
    plot_count = 1 + (1 if 'temperature_mean_mean' in available_cols else 0) + (1 if 'rainfall_total_mean' in available_cols else 0)
    fig, axes = plt.subplots(1, plot_count, figsize=(5*plot_count, 5))
    if plot_count == 1:
        axes = [axes]
    plot_idx = 0
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, square=True, ax=axes[plot_idx])
    axes[plot_idx].set_title('Correlation Matrix', fontsize=12, fontweight='bold')
    plot_idx += 1
    if 'temperature_mean_mean' in available_cols:
        axes[plot_idx].scatter(df_merged['temperature_mean_mean'], df_merged['total_count'], s=100, alpha=0.7, color='#A23B72')
        r_val = corr_matrix.loc['total_count', 'temperature_mean_mean']
        axes[plot_idx].set_title(f'Temp vs Abundance\nr = {r_val:.3f}')
        axes[plot_idx].set_xlabel('Temperature (°C)')
        axes[plot_idx].set_ylabel('Abundance')
        plot_idx += 1
    if 'rainfall_total_mean' in available_cols:
        axes[plot_idx].scatter(df_merged['rainfall_total_mean'], df_merged['total_count'], s=100, alpha=0.7, color='#F18F01')
        r_val = corr_matrix.loc['total_count', 'rainfall_total_mean']
        axes[plot_idx].set_title(f'Rainfall vs Abundance\nr = {r_val:.3f}')
        axes[plot_idx].set_xlabel('Rainfall (mm)')
        axes[plot_idx].set_ylabel('Abundance')
    plt.tight_layout()
    safe_species = species.replace('/', '_').replace(' ', '_')
    filepath = os.path.join(output_path, f'correlation_{safe_species}.png')
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return corr_matrix

def analyze_lagged_correlations(df_merged, max_lag=3, output_path=None, species=None):
    if df_merged.empty or len(df_merged) < max_lag+1:
        return {}
    climate_vars = []
    if 'temperature_mean_mean' in df_merged.columns and not df_merged['temperature_mean_mean'].isna().all():
        climate_vars.append(('temperature_mean_mean', 'Temperature', '#A23B72'))
    if 'rainfall_total_mean' in df_merged.columns and not df_merged['rainfall_total_mean'].isna().all():
        climate_vars.append(('rainfall_total_mean', 'Rainfall', '#F18F01'))
    if not climate_vars:
        return {}
    n_rows = len(climate_vars)
    fig, axes = plt.subplots(n_rows, max_lag, figsize=(15, 5*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    lags = range(1, max_lag+1)
    lag_correlations = {}
    for row_idx, (var_name, var_label, color) in enumerate(climate_vars):
        var_corrs = []
        for i, lag in enumerate(lags):
            df_lag = df_merged[[var_name, 'total_count']].copy()
            df_lag[f'{var_name}_lag'] = df_lag[var_name].shift(lag)
            df_clean = df_lag.dropna()
            if len(df_clean) > 1:
                corr = df_clean[[f'{var_name}_lag', 'total_count']].corr().iloc[0,1]
                if not np.isnan(corr):
                    var_corrs.append(corr)
                    axes[row_idx, i].scatter(df_clean[f'{var_name}_lag'], df_clean['total_count'], alpha=0.7, s=80, color=color)
                    axes[row_idx, i].set_title(f'r = {corr:.3f}\nn={len(df_clean)}')
                    axes[row_idx, i].set_xlabel(f'{var_label} (lag {lag} mo)')
                    axes[row_idx, i].set_ylabel('Abundance')
            else:
                axes[row_idx, i].text(0.5, 0.5, 'Insufficient data', ha='center', va='center')
        lag_correlations[var_label.lower()] = var_corrs
    plt.suptitle('Lagged Correlation Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    if output_path and species:
        safe_species = species.replace('/', '_').replace(' ', '_')
        filepath = os.path.join(output_path, f'lagged_correlation_{safe_species}.png')
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return lag_correlations

def create_sampling_map(gdf_points, region_name, output_path, species):
    if gdf_points is None or len(gdf_points) == 0:
        return None
    df_sites = gdf_points.groupby(['location_code', 'lat', 'lon'], as_index=False)['count'].agg(['sum', 'mean', 'count'])
    df_sites.columns = ['location_code', 'lat', 'lon', 'total_abundance', 'mean_abundance', 'n_observations']
    df_sites = df_sites[df_sites['total_abundance'] > 0]
    gdf_sites = gpd.GeoDataFrame(df_sites, geometry=gpd.points_from_xy(df_sites['lon'], df_sites['lat']), crs='EPSG:4326')
    gdf_sites_3857 = gdf_sites.to_crs(epsg=3857)
    fig, ax = plt.subplots(figsize=(14, 12))
    min_size, max_size = 100, 800
    if df_sites['total_abundance'].max() > 0:
        sizes = min_size + (gdf_sites_3857['total_abundance'] / gdf_sites_3857['total_abundance'].max()) * (max_size - min_size)
    else:
        sizes = min_size
    scatter = ax.scatter(gdf_sites_3857.geometry.x, gdf_sites_3857.geometry.y, s=sizes, c=gdf_sites_3857['mean_abundance'], cmap='plasma', alpha=0.8, edgecolors='white', linewidth=1.5)
    plt.colorbar(scatter, ax=ax, shrink=0.6, label='Mean Abundance')
    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.7)
    except:
        pass
    ax.set_title(f'Vector Sampling Sites - {region_name}\n{species}', fontsize=16, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    safe_species = species.replace('/', '_').replace(' ', '_')
    filepath = os.path.join(output_path, f'spatial_map_{region_name}_{safe_species}.png')
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return gdf_sites

### 6. Core Analysis Function for a Single Group (Fixed)

In [27]:
def analyse_group(country_name, country_code, region_name, species_pattern, start_date, end_date, output_dir):
    """
    Perform full analysis for one country/region/species combination.
    Assumes vector points are linked to region via parent_location_code.
    """
    # 1. Get region_code
    region_code = get_region_code(country_code, region_name)
    if not region_code:
        print(f"  Could not find region_code for {region_name}")
        return None
    
    # 2. Load vector points belonging to this region
    df_vec, gdf_points = load_vector_data_for_region(region_code, species_pattern, start_date, end_date)
    if df_vec.empty:
        return None
    
    # 3. Load environmental data for this region
    df_env = load_environmental_data([region_code], start_date, end_date)
    if df_env.empty:
        return None
    
    # 4. Aggregate vector points to monthly totals (already spatially in region)
    df_vec_monthly = aggregate_to_region_monthly(gdf_points, region_code)
    if df_vec_monthly.empty:
        return None
    
    # 5. Aggregate environmental data to monthly
    df_env_monthly = aggregate_environmental_monthly(df_env, region_code)
    if df_env_monthly.empty:
        return None
    
    # 6. Merge
    df_merged = merge_datasets(df_vec_monthly, df_env_monthly)
    if df_merged.empty:
        return None
    
    # 7. Generate outputs
    os.makedirs(output_dir, exist_ok=True)
    plot_time_series(df_merged, region_name, species_pattern, output_dir)
    corr_matrix = analyze_correlations(df_merged, output_dir, species_pattern)
    lag_corrs = analyze_lagged_correlations(df_merged, max_lag=3, output_path=output_dir, species=species_pattern)
    create_sampling_map(gdf_points, region_name, output_dir, species_pattern)
    
    # 8. Build summary
    summary = {
        'country': country_name,
        'country_code': country_code,
        'region': region_name,
        'species': species_pattern,
        'months': len(df_merged),
        'total_abundance': df_merged['total_count'].sum(),
        'mean_abundance': df_merged['total_count'].mean(),
        'peak_month': df_merged.loc[df_merged['total_count'].idxmax(), 'date_id'],
        'peak_abundance': df_merged['total_count'].max(),
        'date_min': df_merged['date_id'].min(),
        'date_max': df_merged['date_id'].max(),
    }
    if corr_matrix is not None:
        if 'temperature_mean_mean' in corr_matrix.columns:
            summary['corr_temp'] = corr_matrix.loc['total_count', 'temperature_mean_mean']
        if 'rainfall_total_mean' in corr_matrix.columns:
            summary['corr_rain'] = corr_matrix.loc['total_count', 'rainfall_total_mean']
    if lag_corrs:
        for var, vals in lag_corrs.items():
            for i, v in enumerate(vals, 1):
                summary[f'lag_{i}_{var}'] = v
    return summary

### 7. Main Exploratory Loop – Analyse Everything

In [28]:
# Get all available dimensions
countries_df = get_available_countries()
species_list = get_available_species()
global_min_date, global_max_date = get_global_date_range()

print(f"Found {len(countries_df)} countries, {len(species_list)} vector species.")
print(f"Global date range: {global_min_date} to {global_max_date}")

all_results = []

for _, country in countries_df.iterrows():
    country_name = country['country_name']
    country_code = country['country_code']
    print(f"\n--- Processing country: {country_name} ({country_code}) ---")
    regions_df = get_available_regions(country_code)
    if regions_df.empty:
        print(f"  No regions with both environmental data and vector points found for {country_name}")
        continue
    for _, region in regions_df.iterrows():
        region_name = region['name']
        print(f"  Region: {region_name}")
        for species in tqdm(species_list, desc=f"    Species in {region_name}"):
            output_subdir = Path(FIGURE_OUTPUT_PATH) / country_name / region_name / species.replace('/', '_').replace(' ', '_')
            summary = analyse_group(
                country_name, country_code, region_name, species,
                global_min_date, global_max_date, output_subdir
            )
            if summary is not None:
                all_results.append(summary)
                print(f"      ✓ {species}: {summary['months']} months, total abundance {summary['total_abundance']}")

print(f"\n\n=== EXPLORATORY ANALYSIS COMPLETE ===")
print(f"Generated results for {len(all_results)} combinations.")

Found 39 countries, 1 vector species.
Global date range: 1900-06-01 to 2025-12-31

--- Processing country: Angola (AO) ---
  No regions with both environmental data and vector points found for Angola

--- Processing country: Benin (BJ) ---
  No regions with both environmental data and vector points found for Benin

--- Processing country: Botswana (BW) ---
  No regions with both environmental data and vector points found for Botswana

--- Processing country: Burkina Faso (BF) ---
  No regions with both environmental data and vector points found for Burkina Faso

--- Processing country: Burundi (BI) ---
  No regions with both environmental data and vector points found for Burundi

--- Processing country: Cameroon (CM) ---
  No regions with both environmental data and vector points found for Cameroon

--- Processing country: Central African Republic (CF) ---
  No regions with both environmental data and vector points found for Central African Republic

--- Processing country: Comoros (KM

### 8. Export Master Summary Table

In [29]:
if all_results:
    master_df = pd.DataFrame(all_results)
    master_csv_path = os.path.join(FIGURE_OUTPUT_PATH, 'exploratory_analysis_summary.csv')
    master_df.to_csv(master_csv_path, index=False)
    print(f"Master summary saved to: {master_csv_path}")
    print("\nFirst 10 rows of summary:")
    print(master_df.head(10).to_string())
else:
    print("No results to export.")

No results to export.


### 9. (Optional) Interactive Selection for Manual Analysis

In [30]:
# Uncomment and run this cell to interactively choose a single combination for detailed inspection
"""
import ipywidgets as widgets
from IPython.display import display

country_dropdown = widgets.Dropdown(options=countries_df['country_name'].tolist(), description='Country:')
region_dropdown = widgets.Dropdown(description='Region:')
species_dropdown = widgets.Dropdown(options=species_list, description='Species:')
run_button = widgets.Button(description='Run Analysis')
output_area = widgets.Output()

def on_country_change(change):
    selected_country = change.new
    country_code = countries_df[countries_df['country_name']==selected_country]['country_code'].iloc[0]
    regions = get_available_regions(country_code)
    region_dropdown.options = regions['name'].tolist()

country_dropdown.observe(on_country_change, names='value')

def on_run_clicked(b):
    with output_area:
        output_area.clear_output()
        country = country_dropdown.value
        region = region_dropdown.value
        species = species_dropdown.value
        print(f"Analysing {country} / {region} / {species}...")
        country_code = countries_df[countries_df['country_name']==country]['country_code'].iloc[0]
        out_dir = Path(FIGURE_OUTPUT_PATH) / 'manual' / country / region / species.replace('/', '_')
        summary = analyse_group(country, country_code, region, species, global_min_date, global_max_date, out_dir)
        if summary:
            print("Analysis completed. Summary:")
            for k,v in summary.items():
                print(f"  {k}: {v}")
        else:
            print("No data for this combination.")

run_button.on_click(on_run_clicked)
display(country_dropdown, region_dropdown, species_dropdown, run_button, output_area)
"""

'\nimport ipywidgets as widgets\nfrom IPython.display import display\n\ncountry_dropdown = widgets.Dropdown(options=countries_df[\'country_name\'].tolist(), description=\'Country:\')\nregion_dropdown = widgets.Dropdown(description=\'Region:\')\nspecies_dropdown = widgets.Dropdown(options=species_list, description=\'Species:\')\nrun_button = widgets.Button(description=\'Run Analysis\')\noutput_area = widgets.Output()\n\ndef on_country_change(change):\n    selected_country = change.new\n    country_code = countries_df[countries_df[\'country_name\']==selected_country][\'country_code\'].iloc[0]\n    regions = get_available_regions(country_code)\n    region_dropdown.options = regions[\'name\'].tolist()\n\ncountry_dropdown.observe(on_country_change, names=\'value\')\n\ndef on_run_clicked(b):\n    with output_area:\n        output_area.clear_output()\n        country = country_dropdown.value\n        region = region_dropdown.value\n        species = species_dropdown.value\n        print(f"Ana